# P1-26 · Revision checks for Paper 1 (CPU, about 30–45 min, no training)
**Runtime: CPU.** Read-only for `QSEG_FINAL`, all run folders and `mias_ext`. Writes only `MyDrive/P1_v2/revision_checks/`.

1. **Initialization audit.** Every model family is rebuilt with exactly the seed handling of its training code
   (original/fixed-budget runs: `qseg.models.build_model`; E1/E2 and HQB-trig: the model code of P1-17c/17d) and the initial
   tensors of all layers are compared between HQB-UNet and each control (exact equality and maximum absolute difference).
2. **mini-MIAS expert-mask orientation without the labelling rule.** For each expert mask: the NRRD header, and the orientation
   chosen by an image-only criterion (contrast between grey values inside and outside the mask; no threshold, no morphology,
   no proxy mask). Agreement with the orientation used in the paper; overlay sheets for visual verification; left–right
   pairing of consecutive images (one woman per pair).
3. **Re-scoring** of every saved run (stored mini-MIAS probability maps) against the masks aligned by the image-only criterion.
4. **DMID metadata:** files of the figshare record other than the images, to look for patient or examination identifiers.

In [ ]:
# 1 · Drive, paths
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, glob, json, time, shutil, random, gc, re
import numpy as np, pandas as pd, cv2
ROOT  = '/content/drive/MyDrive/QSEG_FINAL'          # read only
OUT   = '/content/drive/MyDrive/P1_v2'
MIAS  = f'{OUT}/mias_ext'
RC    = f'{OUT}/revision_checks'; os.makedirs(RC, exist_ok=True); os.makedirs(f'{RC}/mias_sheets', exist_ok=True)
LOCAL = '/content/qseg_local'
assert not RC.startswith(ROOT)
if not os.path.exists(LOCAL + '/data/manifest.csv'):
    shutil.copytree(ROOT + '/results/data', LOCAL + '/data', ignore=shutil.ignore_patterns('*.npy', '*.h5'))
sys.path.insert(0, ROOT + '/code')
import tensorflow as tf
from tensorflow.keras import layers, models
from qseg.config import Config
from qseg.models import conv_block, output_layer, TFCircuitLayer, build_model
from qseg.datasets import load_manifest, make_dataset
from qseg.data_prep import pad_to_square
cfg = Config(work_dir=LOCAL)
man = load_manifest(cfg)
print('TF', tf.__version__, '| Keras', tf.keras.__version__ if hasattr(tf.keras, '__version__') else '')

In [ ]:
# 3 · Model code of notebooks P1-17c/P1-17d (unchanged)
class CodeSwitch(layers.Layer):
    """Identity during training; at test time can replace the four-value code by supplied codes."""
    def build(self, input_shape):
        self.on = self.add_weight(shape=(), initializer='zeros', trainable=False, name='on')
        self.buf = self.add_weight(shape=(cfg.batch_size, 4), initializer='zeros', trainable=False, name='buf')
    def call(self, z):
        n = tf.shape(z)[0]
        return z * (1.0 - self.on) + self.buf[:n] * self.on

class TrigLayer(layers.Layer):
    """20-parameter trigonometric 4 -> 4 map: z = tanh(A cos((pi/2) x + phi))."""
    def build(self, input_shape):
        self.A = self.add_weight(shape=(4, 4), initializer='glorot_uniform', name='A')
        self.phi = self.add_weight(shape=(4,), initializer=tf.keras.initializers.RandomUniform(0.0, 2 * np.pi), name='phi')
    def call(self, x):
        return tf.tanh(tf.matmul(tf.cos(np.pi / 2 * x + self.phi), self.A, transpose_b=True))

def build(kind, skips=True, filters=(32, 64, 128, 256, 512), dropouts=(0.1, 0.1, 0.2, 0.2, 0.3)):
    inp = layers.Input((256, 256, 1), name='input_image')
    sk, x = [], inp
    for i, (f, d) in enumerate(zip(filters[:-1], dropouts[:-1])):
        x = conv_block(x, f, d, name=f'enc{i + 1}'); sk.append(x)
        x = layers.MaxPooling2D(2, name=f'pool{i + 1}')(x)
    x = conv_block(x, filters[-1], dropouts[-1], name='bottleneck')
    z = layers.Dense(4, activation='tanh', name='pre_dense')(layers.GlobalAveragePooling2D(name='gap')(x))
    if kind == 'quantum':   z = TFCircuitLayer(4, 2, name='quantum_circuit')(z)
    elif kind == 'trig':    z = TrigLayer(name='trig_replacement')(z)
    elif kind == 'classical': z = layers.Dense(4, activation='tanh', name='classical_replacement')(z)
    z = CodeSwitch(name='code_switch')(z)
    q = layers.Dense(512, activation='relu', name='post_dense')(z)
    x = layers.Reshape((16, 16, filters[-1]), name='q_reshape')(
        layers.Dense(16 * 16 * filters[-1], activation='relu', name='q_expand')(q))
    for i, (f, s, d) in enumerate(zip(reversed(filters[:-1]), reversed(sk), reversed(dropouts[:-1]))):
        x = layers.UpSampling2D(2, name=f'up{i + 1}')(x)
        if skips: x = layers.Concatenate(name=f'concat{i + 1}')([x, s])
        x = conv_block(x, f, d, name=f'dec{i + 1}')
    return models.Model(inp, output_layer(x), name=f'HQB_{kind}_{"skip" if skips else "noskip"}')

def set_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)



In [ ]:
# 4 · Initialization audit
ENC = ('enc', 'bottleneck', 'pre_dense')
def part(name):
    return 'encoder' if name.startswith(ENC) else ('mapping' if name in ('quantum_circuit', 'classical_replacement', 'trig_replacement', 'code_switch') else 'decoder')
def snapshot(model):
    return {l.name: [w.numpy().copy() for w in l.weights] for l in model.layers if l.weights}
def make(family, kind, seed):
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.mixed_precision.set_global_policy('float32'); set_seeds(seed)
    make_dataset(cfg, 'train', seed, man); make_dataset(cfg, 'val', seed, man)       # as in the training code
    if family == 'original':
        m = build_model({'quantum': 'hybrid_quantum', 'classical': 'hybrid_classical', 'none': 'hybrid_none'}[kind], cfg)
    else:
        m = build(kind, skips=(family == 'E1'))
    s = snapshot(m); del m; return s
FAM = {'original': ('quantum', 'classical', 'none'), 'E1': ('quantum', 'classical', 'none', 'trig'), 'E2': ('quantum', 'classical', 'none', 'trig')}
rows = []
for family, kinds in FAM.items():
    for seed in range(5):
        S = {k: make(family, k, seed) for k in kinds}
        S['quantum_rebuilt'] = make(family, 'quantum', seed)          # same model twice: must be identical
        ref = S['quantum']
        for other in [k for k in S if k != 'quantum']:
            for name, a in ref.items():
                if name not in S[other]: continue
                b = S[other][name]
                same_shape = len(a) == len(b) and all(x.shape == y.shape for x, y in zip(a, b))
                rows.append(dict(family=family, seed=seed, other=other, layer=name, part=part(name), n=int(sum(x.size for x in a)),
                                 identical=bool(same_shape and all(np.array_equal(x, y) for x, y in zip(a, b))),
                                 max_abs_diff=float(max(np.abs(x - y).max() for x, y in zip(a, b))) if same_shape else np.nan))
        print(family, 'seed', seed, 'done', flush=True)
IA = pd.DataFrame(rows); IA.to_csv(f'{RC}/init_audit_layers.csv', index=False)
summ = (IA.assign(n_same=IA.n * IA.identical).groupby(['family', 'other', 'part'])
          .agg(layers=('layer', 'nunique'), params=('n', 'sum'), params_identical=('n_same', 'sum'), seeds=('seed', 'nunique')))
summ['share_identical'] = (summ.params_identical / summ.params).round(4)
summ.to_csv(f'{RC}/init_audit_summary.csv'); pd.set_option('display.width', 200)
print(summ.to_string())
print('first differing layer (HQB-UNet vs control, seed 0):')
print(IA[(IA.seed == 0) & ~IA.identical].groupby(['family', 'other']).layer.first().to_string())

In [ ]:
# 5 · mini-MIAS images (as in P1-16c; nothing in mias_ext is changed) (Drive copy first, then the PEIPA server) and the expert masks (Zenodo)
!pip -q install pynrrd
import nrrd, tarfile, shutil, requests, glob, zipfile
import matplotlib.pyplot as plt
MR = '/content/mias_raw'; os.makedirs(MR, exist_ok=True)
DRV = '/content/drive/MyDrive/P1_v2/mias_raw'      # put a downloaded mini-MIAS archive (zip or tar.gz) or the .pgm files here
def unpack(path, dst):
    if path.endswith('.zip'): zipfile.ZipFile(path).extractall(dst)
    elif path.endswith(('.tar.gz', '.tgz', '.tar')): tarfile.open(path).extractall(dst)
def find_pgms():
    return sorted(p for p in glob.glob(f'{MR}/**/*.pgm', recursive=True) if os.path.basename(p).lower().startswith('mdb'))
if os.path.isdir(DRV):
    for f in glob.glob(f'{DRV}/**/*', recursive=True):
        if f.lower().endswith('.pgm'): shutil.copy(f, MR)
        elif f.lower().endswith(('.zip', '.tar.gz', '.tgz', '.tar')): unpack(f, MR)
    for _ in range(2):                                      # archives inside archives (e.g. Kaggle zip containing all-mias.tar.gz)
        for f in glob.glob(f'{MR}/**/*', recursive=True):
            if f.lower().endswith(('.tar.gz', '.tgz', '.tar')) and not os.path.exists(f + '.done'):
                unpack(f, os.path.dirname(f)); open(f + '.done', 'w').close()
if len(find_pgms()) < 300:
    for url in ('http://peipa.essex.ac.uk/pix/mias/all-mias.tar.gz', 'https://peipa.essex.ac.uk/pix/mias/all-mias.tar.gz'):
        try:
            r = requests.get(url, timeout=300); r.raise_for_status()
            open(f'{MR}/all-mias.tar.gz', 'wb').write(r.content); tarfile.open(f'{MR}/all-mias.tar.gz').extractall(MR); break
        except Exception as e:
            print('failed', url, type(e).__name__)
pgms = find_pgms(); print(len(pgms), 'mini-MIAS images')
assert len(pgms) >= 300, 'mini-MIAS images not found: put the archive in MyDrive/P1_v2/mias_raw and run again'
import cv2
shapes = {cv2.imread(p, cv2.IMREAD_GRAYSCALE).shape for p in pgms[:20]}
print('image sizes', shapes); assert shapes == {(1024, 1024)}, 'expected the 1024 x 1024 mini-MIAS images'
rec = requests.get('https://zenodo.org/api/records/10149914', timeout=60).json()
for f in rec['files']:
    url = f['links'].get('self') or f['links'].get('download')
    dst = f"{MR}/{f['key']}"
    if not os.path.exists(dst): open(dst, 'wb').write(requests.get(url, timeout=600).content)
    if dst.endswith('.zip'): zipfile.ZipFile(dst).extractall(f'{MR}/masks')
nrrds = sorted(glob.glob(f'{MR}/masks/**/*.nrrd', recursive=True)); print(len(nrrds), 'expert mask files')


In [ ]:
# 6 · Orientation of every expert mask without the labelling rule
mm = pd.read_csv(f'{MIAS}/manifest.csv'); mm = mm[mm.status == 'ok'].drop_duplicates('file_id').set_index('file_id')   # P1-16c listed each image twice
mask_by_id = {}
for p in nrrds:
    m = re.search(r'mdb(\d{3})', os.path.basename(p).lower())
    if m: mask_by_id['mdb' + m.group(1)] = p
pgm_by_id = {os.path.basename(p)[:-4].lower(): p for p in pgms}
def contrast(gray, m):
    m = m > 0
    return float(gray[m].mean() - gray[~m].mean()) if m.any() and (~m).any() else -1e9
rows, H, LAB = [], {}, {}
for fid in mm.index:
    gray = cv2.imread(pgm_by_id[fid], cv2.IMREAD_GRAYSCALE).astype(np.float32)
    lab, hdr = nrrd.read(mask_by_id[fid]); lab = np.squeeze(lab)          # pynrrd default (Fortran index order), as in P1-16c
    cands = {'id': lab, 'T': lab.T, 'ud': lab[::-1], 'Tud': lab.T[::-1], 'lr': lab[:, ::-1], 'Tlr': lab.T[:, ::-1]}
    cands = {k: v for k, v in cands.items() if v.shape == gray.shape}
    sc = {k: contrast(gray, v) for k, v in cands.items()}
    best = max(sc, key=sc.get); second = sorted(sc.values())[-2]
    LAB[fid] = np.ascontiguousarray(cands[best])
    full = (LAB[fid] == 1) | (LAB[fid] == 2); cx = np.nonzero(full)[1].mean() if full.any() else np.nan
    rows.append(dict(file_id=fid, used_in_paper=mm.loc[fid, 'orientation'], image_only=best, contrast_best=round(sc[best], 2),
                     contrast_margin=round(sc[best] - second, 2), contrast_paper=round(sc.get(mm.loc[fid, 'orientation'], np.nan), 2),
                     labels=' '.join(map(str, np.unique(lab).tolist())), chest_wall='left' if cx < gray.shape[1] / 2 else 'right',
                     space=str(hdr.get('space', '')), space_directions=str(hdr.get('space directions', '')).replace('\n', ' '),
                     space_origin=str(hdr.get('space origin', '')).replace('\n', ' '), sizes=str(hdr.get('sizes', ''))))
OR = pd.DataFrame(rows); OR['agree'] = OR.used_in_paper == OR.image_only
OR['woman'] = (OR.file_id.str[3:].astype(int) + 1) // 2
OR.to_csv(f'{RC}/mias_orientation.csv', index=False)
print('orientation used in the paper == image-only criterion:', int(OR.agree.sum()), 'of', len(OR))
print(OR[~OR.agree][['file_id', 'used_in_paper', 'image_only', 'contrast_paper', 'contrast_best']].to_string())
print('orientations used in the paper:', OR.used_in_paper.value_counts().to_dict())
print('NRRD header variants (space directions / origin):'); print(OR.groupby(['space_directions', 'space_origin']).size().to_string())
print('header variants of the masks not stored as "T":'); print(OR[OR.used_in_paper != 'T'][['file_id', 'used_in_paper', 'space_directions', 'space_origin']].to_string())
print('label values:', OR.labels.value_counts().to_dict())
pairs = OR.groupby('woman').chest_wall.agg(lambda s: '/'.join(s))
print('consecutive pairs (one woman) with opposite chest-wall sides:', int(pairs.isin(['left/right', 'right/left']).sum()), 'of', len(pairs))

In [ ]:
# 7 · Overlay sheets for visual verification (expert outline only: green = breast + pectoral, orange = pectoral; no proxy mask)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
def overlay(fid, lab):
    g = cv2.imread(pgm_by_id[fid], cv2.IMREAD_GRAYSCALE); g = cv2.resize(g, (256, 256), interpolation=cv2.INTER_AREA)
    im = cv2.cvtColor(g, cv2.COLOR_GRAY2RGB)
    for m, col in ((((lab == 1) | (lab == 2)), (60, 220, 60)), ((lab == 2), (255, 140, 0))):
        mm_ = cv2.resize(m.astype(np.uint8), (256, 256), interpolation=cv2.INTER_NEAREST)
        cs, _ = cv2.findContours(mm_, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(im, cs, -1, col, 2)
    return im
ids = list(OR.file_id); flag = set(OR[(OR.used_in_paper != 'T') | ~OR.agree].file_id)
for k, p in enumerate(range(0, len(ids), 48)):
    fig, ax = plt.subplots(6, 8, figsize=(16, 12.5))
    for a, fid in zip(ax.ravel(), ids[p:p + 48]):
        a.imshow(overlay(fid, LAB[fid])); a.set_title(f"{fid} [{OR.set_index('file_id').loc[fid, 'image_only']}]", fontsize=7,
                                                     color='red' if fid in flag else 'black')
    for a in ax.ravel(): a.axis('off')
    plt.tight_layout(); plt.savefig(f'{RC}/mias_sheets/sheet_{k + 1}.png', dpi=80); plt.close()
fig, ax = plt.subplots(max(len(flag), 1), 2, figsize=(6, 3 * max(len(flag), 1)), squeeze=False)
for r, fid in enumerate(sorted(flag)):
    lab, _ = nrrd.read(mask_by_id[fid]); lab = np.squeeze(lab)
    ax[r, 0].imshow(overlay(fid, lab.T)); ax[r, 0].set_title(f'{fid}: common transform T', fontsize=8)
    ax[r, 1].imshow(overlay(fid, LAB[fid])); ax[r, 1].set_title(f"{fid}: image-only choice {OR.set_index('file_id').loc[fid, 'image_only']}", fontsize=8)
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.savefig(f'{RC}/mias_sheets/flagged_cases.png', dpi=100); plt.close()
print('sheets:', sorted(os.listdir(f'{RC}/mias_sheets')))

In [ ]:
# 8 · Re-score every saved run against the expert masks aligned by the image-only criterion (same padding and resizing as P1-16c)
side_of = {}
for fid in mm.index:
    s = cv2.imread(f'{MIAS}/images/{fid}.png', 0)                       # stored model input: padding side is visible as zero columns
    side_of[fid] = 'left' if s[:, :8].max() == 0 and s[:, -8:].max() > 0 else ('right' if s[:, -8:].max() == 0 and s[:, :8].max() > 0 else 'center')
def to256(m, side):
    return cv2.resize(pad_to_square(m.astype(np.uint8) * 255, side), (256, 256), interpolation=cv2.INTER_NEAREST) > 127
NEW = {fid: dict(full=to256((LAB[fid] == 1) | (LAB[fid] == 2), side_of[fid]), breast=to256(LAB[fid] == 1, side_of[fid])) for fid in mm.index}
OLD = {fid: dict(full=cv2.imread(f'{MIAS}/masks_expert/{fid}.png', 0) > 127, breast=cv2.imread(f'{MIAS}/masks_expert_breast/{fid}.png', 0) > 127) for fid in mm.index}
ch = pd.DataFrame([dict(file_id=f, mask_changed=bool((NEW[f]['full'] != OLD[f]['full']).any() or (NEW[f]['breast'] != OLD[f]['breast']).any()),
                        padding_side=side_of[f]) for f in mm.index])
print('padding side found from the stored inputs:', ch.padding_side.value_counts().to_dict(), '| masks that change:', int(ch.mask_changed.sum()))
def dice(a, b):
    s = a.sum() + b.sum(); return 1.0 if s == 0 else 2 * (a & b).sum() / s
rows = []
for d in sorted(glob.glob(f'{OUT}/runs/*_seed*')):
    f = f'{d}/mias_prob_uint8.npz'
    if not os.path.exists(f): continue
    z = np.load(f, allow_pickle=True); P = z['prob'] >= 128; fids = [str(x) for x in z['file_id']]
    name = os.path.basename(d); exp, mdl, s = name.split('_')
    for k, fid in enumerate(fids):
        rows.append(dict(run=name, exp=exp, model=mdl, seed=int(s[4:]), file_id=fid,
                         dice_expert_paper=dice(P[k], OLD[fid]['full']), dice_expert_breast_paper=dice(P[k], OLD[fid]['breast']),
                         dice_expert_imageonly=dice(P[k], NEW[fid]['full']), dice_expert_breast_imageonly=dice(P[k], NEW[fid]['breast'])))
RS = pd.DataFrame(rows); RS.to_csv(f'{RC}/mias_rescored_per_image.csv.gz', index=False)
RS['diff'] = RS.dice_expert_imageonly - RS.dice_expert_paper
print('runs re-scored:', RS.run.nunique(), '| max |change| per image:', round(RS['diff'].abs().max(), 6))
print(RS.groupby(['exp', 'model'])[['dice_expert_paper', 'dice_expert_imageonly', 'dice_expert_breast_paper', 'dice_expert_breast_imageonly']].mean().round(4).to_string())

In [ ]:
# 9 · DMID: files of the figshare record other than the images (patient or examination identifiers?)
import requests, io
art = requests.get('https://api.figshare.com/v2/articles/24522883', timeout=60).json()
FL = pd.DataFrame([dict(name=f['name'], size_mb=round(f['size'] / 2 ** 20, 2)) for f in art['files']]); FL.to_csv(f'{RC}/dmid_record_files.csv', index=False)
print(FL.to_string())
for f in art['files']:
    if 'TIFF' in f['name'] or f['size'] > 50 * 2 ** 20: continue
    b = requests.get(f['download_url'], timeout=300).content; print('\n=====', f['name'])
    try:
        if f['name'].lower().endswith(('.xlsx', '.xls')):
            for sh, df in pd.read_excel(io.BytesIO(b), sheet_name=None).items(): print('sheet', sh, df.shape); print(df.head(8).to_string())
        elif f['name'].lower().endswith('.zip'):
            import zipfile; zf = zipfile.ZipFile(io.BytesIO(b)); names = zf.namelist(); print(len(names), 'files, e.g.', names[:10])
            for n in names:
                if n.lower().endswith(('.txt', '.csv')): print('---', n); print(zf.read(n).decode('utf-8', 'replace')[:1500]); break
        else:
            print(b[:1500].decode('utf-8', 'replace'))
    except Exception as e:
        print('could not read:', e)

In [ ]:
# 10 · Summary
S = dict(init_audit=summ.reset_index().to_dict('records'),
         mias_orientation_agree=int(OR.agree.sum()), mias_n=len(OR), mias_disagree=OR[~OR.agree].file_id.tolist(),
         mias_pairs_opposite=int(pairs.isin(['left/right', 'right/left']).sum()), mias_pairs=len(pairs),
         rescore_max_abs_change=float(RS['diff'].abs().max()), masks_changed=int(ch.mask_changed.sum()))
json.dump(S, open(f'{RC}/summary.json', 'w'), indent=1, default=str)
print(json.dumps({k: v for k, v in S.items() if k != 'init_audit'}, indent=1, default=str))
print('DONE')